# Experiment 031: continuation of the stopped N=4 run, seed 1 (Colab)

This run resumes the copied checkpoint-100 state from the stopped 030 seed 1 arm. The step-150 a2 decision is recorded before training can continue. A reproduced stop continues to step 500; a non-reproduced stop ends at step 150 without endpoint evaluation.

## 0. Compute units and mount Drive

In [1]:
# record the balance before the session starts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 575.43       # fill from the resources panel
unit_rate = 6.77      # units per hour for this card
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '031_colab_b7_n4_a2_record',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f)
        print('opened session record')


session 20260813_111301 | cu_before = 575.43
Mounted at /content/drive
closed previous session: 20260812_225739
opened session record


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime used by the 013 reference arms'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-ccc9c259-55cc-a074-5921-8e1384b159a2)
total vram: 79.3 GiB


## 2. Clone or update the repo

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3896, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 3896 (delta 96), reused 160 (delta 49), pack-reused 3674 (from 1)
Receiving objects: 100% (3896/3896), 117.38 MiB | 18.31 MiB/s, done.
Resolving deltas: 100% (2495/2495), done.
Updating files: 100% (1802/1802), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.3 MB/s eta 0:00:00


## 4. W&B key

In [5]:
import getpass, os
key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
except Exception:
    key = None
if not key and os.path.exists('wandb_api_key.txt'):
    key = open('wandb_api_key.txt').read().strip()
if not key:
    key = getpass.getpass('wandb api key: ').strip()
assert key, 'a wandb key is required because the worker logs online'
os.environ['WANDB_API_KEY'] = key
with open('wandb_api_key.txt', 'w') as f:
    f.write(key)
key = None
print('wandb key set')


wandb key set


## 5. Preflight before any test-set contact

In [6]:
import json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from config import load_config, run_tag
from scripts.s12_eval import atomic_write_json, config_hash_status, sha256
from scripts.s28_compose import (concluded_a2_state, gate_state, merged_hashes,
                                 resolve_source_adapter, verify_historical_merge)
from scripts.s31_sentinel import source_prefix_facts

D = '/content/drive/MyDrive/inm363'
CFG_PATH = 'configs/031_colab_b7_n4_a2_record.yaml'
SOURCE_CFG_PATH = 'configs/030_colab_b7_n4_seed1.yaml'
PARENT_CFG_PATH = 'configs/013_colab_full_b7_seed1.yaml'
SOURCE_TAG = '030_COLAB_FULL_B7_external_sft_grpo_seed1'
PARENT_TAG = '013_COLAB_FULL_B7_sft_grpo_seed1'
HISTORICAL_MANIFEST = os.path.join(
    D, 'results', '022_smooth_B7_seed1_step1000', 'arm_manifest.json')

cfg = load_config(CFG_PATH)
source_cfg = load_config(SOURCE_CFG_PATH)
parent_cfg = load_config(PARENT_CFG_PATH)
TAG = run_tag(cfg)
assert cfg.experiment == '031' and cfg.train.seed == 1
assert cfg.train.a2_action == 'record' and cfg.train.num_generations == 4
assert cfg.condition.name == 'EXTERNAL_SFT_GRPO'

SOURCE_RESULTS = os.path.join(D, 'results', SOURCE_TAG)
SOURCE_OUT = os.path.join(D, 'checkpoints', SOURCE_TAG, 'grpo')
SOURCE_CHECKPOINT = os.path.join(SOURCE_OUT, 'checkpoint-100')
ARCHIVED_HISTORY = os.path.join(SOURCE_RESULTS, 'grpo_trainlog.json')
PARENT_RESULTS = os.path.join(D, 'results', PARENT_TAG)
ARCHIVE_MERGED = os.path.join(D, 'checkpoints', PARENT_TAG, 'sft_merged')
OUT_DIR = os.path.join(D, 'checkpoints', TAG, 'grpo')
MERGED_DIR = os.path.join(D, 'checkpoints', TAG, 'sft_merged')
RES_DIR = os.path.join(D, 'results', TAG)

adir, adapter_hashes = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
parent_manifest = json.load(open(os.path.join(PARENT_RESULTS, 'manifest.json')))
stored_hash = open(os.path.join(adir, 'config_hash.txt')).read().strip()
hash_status = config_hash_status(parent_cfg, parent_manifest, stored_hash)
assert hash_status in ('current', 'legacy_missing_grpo_from_adapter')

tracked_source = os.path.join('results', SOURCE_TAG)
for name in ('arm_manifest.json', 'concluded_a2.json', 'grpo_trainlog.json',
             'parent_gate.json'):
    drive_path = os.path.join(SOURCE_RESULTS, name)
    tracked_path = os.path.join(tracked_source, name)
    assert os.path.exists(drive_path) and os.path.exists(tracked_path),         f'missing stopped-arm artifact: {name}'
    assert sha256(drive_path) == sha256(tracked_path),         f'stopped-arm artifact changed: {name}'

source_arm = json.load(open(os.path.join(SOURCE_RESULTS, 'arm_manifest.json')))
assert concluded_a2_state(SOURCE_RESULTS, source_arm['attempts']) == 'committed'
source_env = source_arm['attempts'][-1]['env']

prefix = source_prefix_facts(
    SOURCE_OUT, SOURCE_CHECKPOINT, source_cfg, ARCHIVED_HISTORY)
historical = verify_historical_merge(
    ARCHIVE_MERGED, HISTORICAL_MANIFEST, PARENT_TAG)
archive_hashes = merged_hashes(ARCHIVE_MERGED)
source_gate = os.path.join(SOURCE_RESULTS, 'parent_gate.json')
archived_parent_preds = os.path.join(PARENT_RESULTS, 'sft_logprob_preds.jsonl')
assert gate_state(source_gate, adapter_hashes, archived_parent_preds) == 'committed'

record = {
    'config': CFG_PATH, 'tag': TAG, 'source_tag': SOURCE_TAG,
    'parent_tag': PARENT_TAG, 'parent_hash_status': hash_status,
    'adapter_sha256': adapter_hashes, 'source_prefix': prefix,
    'archive_source': ARCHIVE_MERGED, 'archive_sha256': archive_hashes,
    'historical_merge': historical, 'source_gate_sha256': sha256(source_gate),
    'archived_parent_preds_sha256': sha256(archived_parent_preds),
    'source_attempt_env': source_env,
}
preflight_path = os.path.join(D, 'results', '031_preflight.json')
if os.path.exists(preflight_path):
    assert json.load(open(preflight_path)) == record, '031 preflight sources changed'
else:
    atomic_write_json(preflight_path, record)

free_gb = shutil.disk_usage(D).free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 45, 'need 45GB free for the copied base and continuation checkpoints'
print('preflight green:', TAG)


drive free: 164 GB
preflight green: 031_COLAB_FULL_B7_external_sft_grpo_seed1


## 6. Wiring checks

In [7]:
import subprocess, sys
r = subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_031_sentinel_validation.py',
    'tests/test_checkpoint.py',
    'tests/test_grpo_worker.py',
    'tests/test_s28_composition.py', '-q'])
assert r.returncode == 0, 'package tests failed, nothing runs'

from scripts.grpo_worker import (active_worker_processes, blocked_resume_reason,
                                 stream_subprocess)
from scripts.s12_eval import IntegrityError, read_jsonl
from scripts.s28_compose import (arm_state, commit_endpoint_eval,
                                 commit_training_complete, endpoint_state,
                                 stage_archived_merge, training_state,
                                 verify_archived_merge, write_arm_complete)
from scripts.s31_sentinel import (commit_non_reproducing, non_reproducing_state,
                                  require_true_verdict, reuse_parent_gate,
                                  stage_checkpoint, verdict_state,
                                  verify_staged_checkpoint)
from obs import _git_info, _versions, append_ledger, gpu_snapshot
from train.model_utils import resolve_dtype
print('package imports fine')


package imports fine


In [8]:
!pip uninstall -q -y torchao


## 7. Build the data splits

In [9]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 26.2MB/s]

main/train-00000-of-00001.parquet: downloading bytes:  87% 2.01M/2.31M [00:01<00:00, 1.41MB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.41MB/s,  220kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.42MB/s,  223kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 316kB/s, 40.7kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 316kB/s, 40.7kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 423402.10 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 293786.15 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 87.0MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 132MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00

## 8. Reuse the archived base evaluation

In [10]:
from scripts.s28_compose import prepopulate_base_evals
base_source = os.path.join(D, 'results', 'base_evals', '013_COLAB_FULL_B7')
base_target = os.path.join(D, 'results', 'base_evals', '031_COLAB_FULL_B7')
base_record = prepopulate_base_evals(base_source, base_target)
print('base reuse covers', sorted(base_record['sha256']))


  base evals prepopulated from 013_COLAB_FULL_B7
base reuse covers ['base_eval.json', 'base_logprob.json', 'base_logprob_preds.jsonl', 'base_preds.jsonl']


## 9. Stage the exact starting state

In [11]:
os.makedirs(RES_DIR, exist_ok=True)
verify_historical_merge(ARCHIVE_MERGED, HISTORICAL_MANIFEST, PARENT_TAG)
merge_record = stage_archived_merge(
    ARCHIVE_MERGED, MERGED_DIR, adir, OUT_DIR)
verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, adir)

stage_record = stage_checkpoint(
    SOURCE_OUT, SOURCE_CHECKPOINT, OUT_DIR, source_cfg, cfg, ARCHIVED_HISTORY)
verify_staged_checkpoint(
    SOURCE_OUT, SOURCE_CHECKPOINT, OUT_DIR, source_cfg, cfg, ARCHIVED_HISTORY)
print('merged files:', len(merge_record['destination_sha256']))
print('checkpoint files:', len(stage_record['destination_checkpoint_sha256']))


  archived 013 merged base copied and committed
merged files: 6
checkpoint files: 11


## 10. Reuse the completed parent gate

In [12]:
GATE_REUSE = os.path.join(RES_DIR, 'parent_gate_reuse.json')
gate_record = reuse_parent_gate(
    source_gate, GATE_REUSE, adapter_hashes, archived_parent_preds)
print('parent gate reused:', gate_record['stats'])

free_gib = torch.cuda.mem_get_info()[0] / 1024**3
print(f'gpu free: {free_gib:.1f} GiB')
assert free_gib >= 70, 'need 70 GiB free before 7b training'


parent gate reused: {'agreement': 1.0, 'acc_new': 0.801486, 'acc_archived': 0.801486}
gpu free: 78.8 GiB


## 11. Continue training and evaluate only if the verdict reproduces

In [13]:
assert cu_before is not None and unit_rate is not None,     'fill cu_before and unit_rate in the units cell'
import glob as _glob, shutil as _shutil, subprocess, sys, time

EVAL_SCRIPT = '/content/s31_eval.py'
with open(EVAL_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from checkpoint import save_metrics
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_eval, run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
device = resolve_device(cfg.env)
ev = run_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_eval.json"), ev)
lp = run_logprob_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_logprob.json"), lp)
''')

os.environ['WANDB_PROJECT'] = cfg.log.wandb_project
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')
MANIFEST_PATH = os.path.join(RES_DIR, 'arm_manifest.json')
TRAINLOG_PATH = os.path.join(RES_DIR, 'grpo_trainlog.json')
allowed_infra_files = {
    'checkpoint.py', 'config.py', 'obs.py', 'train/grpo.py',
    'scripts/s12_eval.py', 'scripts/s31_sentinel.py',
    'configs/031_colab_b7_n4_a2_record.yaml',
    'notebooks/031_colab_b7_n4_a2_record.ipynb',
    'tests/test_031_sentinel_validation.py',
    'tests/test_checkpoint.py', 'tests/test_config.py',
    'tests/test_protocol_freeze.py', 'tests/test_sentinel.py',
    'tests/test_s12_sweep_end_evals.py',
}

for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate',
            'cais/mmlu@', 'openai/gsm8k@'):
    got = _versions().get(lib)
    parent_want = parent_manifest.get('versions', {}).get(lib)
    source_want = source_env.get('versions', {}).get(lib)
    if parent_want is not None:
        assert got == parent_want, f'013 {lib}: {got} vs {parent_want}'
    if source_want is not None:
        assert got == source_want, f'030 {lib}: {got} vs {source_want}'
assert resolve_dtype(cfg) == parent_manifest['resolved_dtype'], 'dtype differs from 013'
assert resolve_dtype(cfg) == source_env['resolved_dtype'], 'dtype differs from 030'
assert cfg.exec.allow_oom_fallback is False
assert (cfg.exec.grpo_batch_size or cfg.train.num_generations) == 4
assert cfg.exec.grpo_grad_accum == 1 and cfg.exec.gradient_checkpointing is False
assert cfg.log.wandb_project == 'inm363-grpo'
verify_historical_merge(ARCHIVE_MERGED, HISTORICAL_MANIFEST, PARENT_TAG)
verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, adir)
verify_staged_checkpoint(
    SOURCE_OUT, SOURCE_CHECKPOINT, OUT_DIR, source_cfg, cfg, ARCHIVED_HISTORY)

def newest_step():
    if os.path.exists(TRAINLOG_PATH):
        rows = json.load(open(TRAINLOG_PATH))
        logged = [int(row['step']) for row in rows if row.get('step') is not None]
        if logged:
            return max(logged)
    checkpoints = _glob.glob(os.path.join(OUT_DIR, 'checkpoint-*'))
    return max((int(path.rsplit('-', 1)[1]) for path in checkpoints), default=0)

if non_reproducing_state(RES_DIR, OUT_DIR, cfg) == 'committed':
    print('non-reproducing outcome already committed, no endpoint runs')
elif arm_state(RES_DIR, OUT_DIR) == 'committed':
    require_true_verdict(OUT_DIR, cfg, TRAINLOG_PATH)
    assert training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'committed'
    print('completed endpoint already committed, skipping')
else:
    current_verdict = verdict_state(OUT_DIR, cfg, TRAINLOG_PATH)
    if current_verdict == 'committed_false':
        commit_non_reproducing(RES_DIR, OUT_DIR, cfg)
        print('step-150 verdict did not reproduce the stop, no endpoint runs')
    else:
        manifest = json.load(open(MANIFEST_PATH)) if os.path.exists(MANIFEST_PATH) else {'attempts': []}
        start_step = newest_step()
        for attempt in manifest.get('attempts', []):
            if 'ts_end' not in attempt:
                attempt.update({
                    'interrupted': True, 'end_step': start_step,
                    'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
        atomic_write_json(MANIFEST_PATH, manifest)

        reason = blocked_resume_reason(manifest.get('attempts', []))
        assert reason is None, reason
        assert not active_worker_processes(CFG_PATH), 'a training worker is already running'

        final_artifacts = [
            os.path.join(OUT_DIR, 'adapter_model.safetensors'),
            os.path.join(OUT_DIR, 'adapter_base.json'),
            TRAINLOG_PATH,
        ]
        trained = all(os.path.exists(path) for path in final_artifacts)
        if start_step >= cfg.train.max_steps and not trained:
            raise IntegrityError(
                'final checkpoint exists but the continuation artifacts are incomplete')
        if not trained:
            env = {'git': _git_info(), 'versions': _versions(),
                   'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
            repo_top = subprocess.run(
                ['git', 'rev-parse', '--show-toplevel'],
                capture_output=True, text=True, check=True).stdout.strip()
            repo_prefix = os.path.relpath(os.getcwd(), repo_top)
            for attempt in manifest.get('attempts', []):
                previous = attempt.get('env')
                if not previous:
                    continue
                old_commit = previous['git'].get('commit')
                new_commit = env['git'].get('commit')
                if old_commit != new_commit:
                    changed = subprocess.run(
                        ['git', 'diff', '--name-only', old_commit, new_commit],
                        capture_output=True, text=True, check=True).stdout.splitlines()
                    prefix = repo_prefix + os.sep
                    changed = [
                        name[len(prefix):]
                        if repo_prefix != '.' and name.startswith(prefix) else name
                        for name in changed]
                    unexpected = sorted(set(changed) - allowed_infra_files)
                    assert not unexpected, f'scientific code drifted: {unexpected}'
                for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
                    assert previous['versions'].get(lib) == env['versions'].get(lib),                         f'{lib} drifted between attempts'
                assert previous['resolved_dtype'] == env['resolved_dtype'],                     'dtype drifted between attempts'

            free_gib = torch.cuda.mem_get_info()[0] / 1024**3
            assert free_gib >= 70, f'only {free_gib:.1f} GiB free before training'
            verify_historical_merge(ARCHIVE_MERGED, HISTORICAL_MANIFEST, PARENT_TAG)
            verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, adir)
            verify_staged_checkpoint(
                SOURCE_OUT, SOURCE_CHECKPOINT, OUT_DIR, source_cfg, cfg,
                ARCHIVED_HISTORY)

            attempt = {
                'start_step': start_step,
                'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                'gpu': gpu_snapshot(), 'env': env,
                'source_checkpoint': '030 seed 1 checkpoint-100',
                'completed': False,
            }
            manifest['attempts'].append(attempt)
            atomic_write_json(MANIFEST_PATH, manifest)

            status_path = '/content/031_worker.json'
            if os.path.exists(status_path):
                os.remove(status_path)
            worker_cmd = [
                sys.executable, '-u', 'scripts/grpo_worker.py',
                '--config', CFG_PATH, '--adapter', adir,
                '--out-dir', OUT_DIR, '--log-dir', RES_DIR,
                '--run-name', f'{TAG}_grpo', '--status', status_path,
                '--report-to', 'wandb',
            ]
            t0 = time.time()
            proc = None
            parent_failure = None
            try:
                print(f'===== continuing from step {start_step} =====')
                proc = stream_subprocess(worker_cmd)
                if proc.returncode != 0:
                    raise RuntimeError(
                        f'training worker failed with exit code {proc.returncode}')
            except BaseException as exc:
                parent_failure = {
                    'error_type': type(exc).__name__,
                    'error_message': str(exc)[:2000],
                }
                raise
            finally:
                wall = time.time() - t0
                end_step = newest_step()
                worker_returncode = proc.returncode if proc is not None else None
                worker_status = (
                    json.load(open(status_path)) if os.path.exists(status_path) else {})
                worker_peak = worker_status.get('peak_vram_gib')
                error_type = (
                    worker_status.get('error_type')
                    or (parent_failure or {}).get('error_type'))
                error_message = (
                    worker_status.get('error_message')
                    or (parent_failure or {}).get('error_message'))
                manual_interrupt = error_type == 'KeyboardInterrupt'
                explicit_failure = (
                    (worker_returncode not in (None, 0) or parent_failure is not None)
                    and not manual_interrupt)
                done = (
                    worker_status.get('completed') is True
                    and all(os.path.exists(path) for path in final_artifacts))
                attempt.update({
                    'end_step': end_step,
                    'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                    'elapsed_sec': round(wall, 1), 'completed': done,
                    'peak_vram_gib': worker_peak,
                    'worker_returncode': worker_returncode,
                    'error_type': error_type, 'error_message': error_message,
                    'cuda_oom': bool(worker_status.get('cuda_oom', False)),
                    'interrupted': manual_interrupt,
                    'resume_blocked': explicit_failure,
                })
                atomic_write_json(MANIFEST_PATH, manifest)
                gpu_row = gpu_snapshot()
                if worker_peak is not None:
                    gpu_row['peak_vram_gb'] = worker_peak
                append_ledger(LEDGER, {
                    'run_tag': TAG,
                    'condition': 'EXTERNAL_SFT_GRPO_train',
                    'timings_sec': {'grpo_train': round(wall, 1)},
                    'train_steps': max(0, end_step - start_step),
                    'completed': done, 'gpu': gpu_row,
                })
            assert worker_status.get('completed') is True,                 'worker status did not complete'
            assert attempt['completed'] is True,                 'continuation artifacts are incomplete'
            print(f'training returned in {wall/3600:.2f}h, peak {worker_peak} GiB')
        else:
            print('continuation adapter already saved')

        current_verdict = verdict_state(OUT_DIR, cfg, TRAINLOG_PATH)
        if current_verdict == 'committed_false':
            commit_non_reproducing(RES_DIR, OUT_DIR, cfg)
            print('step-150 verdict did not reproduce the stop, no endpoint runs')
        else:
            assert current_verdict == 'committed_true'
            require_true_verdict(OUT_DIR, cfg, TRAINLOG_PATH)
            if training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'absent':
                commit_training_complete(RES_DIR, OUT_DIR, MERGED_DIR, cfg)
            assert training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'committed'

            ep_state = endpoint_state(RES_DIR, OUT_DIR)
            if ep_state == 'tampered':
                raise IntegrityError('endpoint files contradict their manifest')
            if ep_state != 'committed':
                require_true_verdict(OUT_DIR, cfg, TRAINLOG_PATH)
                scratch = '/content/s31_eval'
                _shutil.rmtree(scratch, ignore_errors=True)
                t0 = time.time()
                eval_done = False
                try:
                    r = subprocess.run([
                        sys.executable, EVAL_SCRIPT, '--config', CFG_PATH,
                        '--adapter', OUT_DIR, '--out', scratch])
                    assert r.returncode == 0, 'endpoint evaluation failed'
                    eval_wall = round(time.time() - t0, 1)
                    extra = {
                        'adapter': OUT_DIR,
                        'source_adapter': adir,
                        'source_adapter_sha256': adapter_hashes,
                        'archived_merge': verify_archived_merge(
                            ARCHIVE_MERGED, MERGED_DIR, adir),
                        'historical_merge': verify_historical_merge(
                            ARCHIVE_MERGED, HISTORICAL_MANIFEST, PARENT_TAG),
                        'parent_gate_reuse': json.load(open(GATE_REUSE)),
                        'checkpoint_stage': json.load(open(
                            os.path.join(OUT_DIR, '031_checkpoint_stage.json'))),
                        'a2_verdict': require_true_verdict(
                            OUT_DIR, cfg, TRAINLOG_PATH),
                        'training_complete': json.load(open(
                            os.path.join(RES_DIR, 'training_complete.json'))),
                        'config': CFG_PATH, 'seed': 1, 'git': _git_info(),
                        'versions': _versions(),
                        'resolved_dtype': resolve_dtype(cfg),
                        'gpu': gpu_snapshot(),
                        'timings_sec': {'grpo_eval': eval_wall},
                    }
                    commit_endpoint_eval(
                        RES_DIR, scratch, adapter_dir=OUT_DIR,
                        extra_manifest=extra, package='031_sentinel_validation')
                    _shutil.rmtree(scratch, ignore_errors=True)
                    eval_done = True
                finally:
                    append_ledger(LEDGER, {
                        'run_tag': TAG,
                        'condition': 'EXTERNAL_SFT_GRPO_eval',
                        'timings_sec': {
                            'grpo_eval': round(time.time() - t0, 1)},
                        'train_steps': 0, 'completed': eval_done,
                        'gpu': gpu_snapshot(),
                    })

            require_true_verdict(OUT_DIR, cfg, TRAINLOG_PATH)
            r = subprocess.run([
                sys.executable, 'scripts/run.py', '--config', CFG_PATH,
                '--stage', 'all', '--no-ledger'])
            assert r.returncode == 0, 'summary stage failed'
            summary = json.load(open(os.path.join(RES_DIR, 'summary.json')))
            assert summary['reference'] == f'gsm8k_sft_parent:{PARENT_TAG}'
            write_arm_complete(RES_DIR)
            print('gen per-tier:', summary['grpo_acc'])


Streaming output truncated to the last 5000 lines.
│ │ A. All        │               │               │              │           │ │
│ │ behavior can  │               │               │              │           │ │
│ │ be traced to  │               │               │              │           │ │
│ │ human         │               │               │              │           │ │
│ │ biological    │               │               │              │           │ │
│ │ functions.    │               │               │              │           │ │
│ │ B. People's   │               │               │              │           │ │
│ │ behavior is   │               │               │              │           │ │
│ │ primarily a   │               │               │              │           │ │
│ │ result of     │               │               │              │           │ │
│ │ free will.    │               │               │              │           │ │
│ │ C. Behavior   │               │               │       

## 12. Registered comparison

In [14]:
from analysis.stats import _read_preds, align_pred_rows, mcnemar_exact, paired_delta_ci
from data.subjects import TIERS, tier_of

COMPARISON_PATH = os.path.join(D, 'results', '031_comparison_summary.json')

def paired_stats(old_rows, new_rows, seed=1):
    old_aligned, new_aligned = align_pred_rows(old_rows, new_rows)
    old_correct = [row['correct'] for row in old_aligned]
    new_correct = [row['correct'] for row in new_aligned]
    delta, lo, hi = paired_delta_ci(old_correct, new_correct, seed=seed)
    gain = sum(1 for old, new in zip(old_correct, new_correct) if not old and new)
    loss = sum(1 for old, new in zip(old_correct, new_correct) if old and not new)
    return {
        'reference_acc': sum(old_correct) / len(old_correct),
        'continued_acc': sum(new_correct) / len(new_correct),
        'delta': delta, 'ci': [lo, hi],
        'mcnemar_p': mcnemar_exact(gain, loss),
        'discordant': {'continued_gain': gain, 'reference_gain': loss},
    }

def category(stats):
    if abs(stats['delta']) >= 0.02 and stats['mcnemar_p'] < 0.05:
        return 'material improvement' if stats['delta'] > 0 else 'material decline'
    if stats['ci'][0] > -0.02 and stats['ci'][1] < 0.02:
        return 'equivalent'
    return 'inconclusive'

if non_reproducing_state(RES_DIR, OUT_DIR, cfg) == 'committed':
    comparison = {
        'experiment': '031', 'seed': 1,
        'status': 'non_reproducing',
        'registered_reading': 'the continued trajectory did not reproduce the stop; the counterfactual is unaudited',
        'a2_verdict': json.load(open(os.path.join(OUT_DIR, 'a2_verdict.json'))),
        'endpoint_contact': 0,
    }
    atomic_write_json(COMPARISON_PATH, comparison)
elif arm_state(RES_DIR, OUT_DIR) == 'committed':
    require_true_verdict(OUT_DIR, cfg, TRAINLOG_PATH)
    start_gen = _read_preds(os.path.join(PARENT_RESULTS, 'sft_preds.jsonl'))
    final_gen = _read_preds(os.path.join(RES_DIR, 'grpo_preds.jsonl'))
    overall = paired_stats(start_gen, final_gen)
    outcome = category(overall)

    start_aligned, final_aligned = align_pred_rows(start_gen, final_gen)
    tiers = {}
    for tier in TIERS:
        old = [row for row in start_aligned if tier_of(row['subject']) == tier]
        new = [row for row in final_aligned if tier_of(row['subject']) == tier]
        tiers[tier] = paired_stats(old, new)

    start_probe = _read_preds(os.path.join(PARENT_RESULTS, 'sft_logprob_preds.jsonl'))
    final_probe = _read_preds(os.path.join(RES_DIR, 'grpo_logprob_preds.jsonl'))
    probe = paired_stats(start_probe, final_probe)

    history = json.load(open(TRAINLOG_PATH))
    windows = {}
    for label, low, high in (
            ('steps_1_50', 1, 50), ('steps_51_100', 51, 100),
            ('steps_101_150', 101, 150), ('steps_151_500', 151, 500)):
        rows = [row for row in history
                if low <= int(row.get('step', 0)) <= high]
        windows[label] = {}
        for key in ('rewards/accuracy_reward/mean', 'rewards/format_reward/mean',
                    'frac_reward_zero_std', 'reward_std', 'kl'):
            values = [float(row[key]) for row in rows if row.get(key) is not None]
            windows[label][key] = sum(values) / len(values) if values else None

    n8_rows = _read_preds(os.path.join(PARENT_RESULTS, 'grpo_preds.jsonl'))
    n8_overall = sum(row['correct'] for row in n8_rows) / len(n8_rows)
    comparison = {
        'experiment': '031', 'seed': 1, 'status': 'completed',
        'primary_endpoint': 'overall generative accuracy versus the seed-matched GSM8K-SFT start',
        'registered_outcome': outcome,
        'overall': overall, 'tiers_descriptive': tiers,
        'probe_descriptive': probe,
        'n8_twin_overall_descriptive': n8_overall,
        'format_rate': {
            'start': sum(row.get('format_ok', False) for row in start_aligned) / len(start_aligned),
            'continued': sum(row.get('format_ok', False) for row in final_aligned) / len(final_aligned),
        },
        'a2_verdict': require_true_verdict(OUT_DIR, cfg, TRAINLOG_PATH),
        'training_windows': windows,
        'endpoint_contact': 2,
    }
    atomic_write_json(COMPARISON_PATH, comparison)
else:
    comparison = {'experiment': '031', 'status': 'not_completed'}
    print('comparison waits for a committed outcome')

print(json.dumps(comparison, indent=2))


{
  "experiment": "031",
  "seed": 1,
  "status": "completed",
  "primary_endpoint": "overall generative accuracy versus the seed-matched GSM8K-SFT start",
  "registered_outcome": "material improvement",
  "overall": {
    "reference_acc": 0.7786624203821656,
    "continued_acc": 0.804140127388535,
    "delta": 0.025477707006369428,
    "ci": [
      0.012738853503184714,
      0.03821656050955414
    ],
    "mcnemar_p": 0.00012227884454768217,
    "discordant": {
      "continued_gain": 100,
      "reference_gain": 52
    }
  },
  "tiers_descriptive": {
    "easy": {
      "reference_acc": 0.8782707622298066,
      "continued_acc": 0.8930602957906713,
      "delta": 0.01478953356086462,
      "ci": [
        0.0022753128555176336,
        0.02844141069397042
      ],
      "mcnemar_p": 0.040959591511636986,
      "discordant": {
        "continued_gain": 24,
        "reference_gain": 11
      }
    },
    "medium": {
      "reference_acc": 0.7536945812807881,
      "continued_acc": 0.

## 13. Peek at the result

In [15]:
if os.path.exists(COMPARISON_PATH):
    result = json.load(open(COMPARISON_PATH))
    print('status:', result['status'])
    print('registered outcome:', result.get('registered_outcome'))
    if result['status'] == 'completed':
        print('overall delta:', result['overall']['delta'])
        print('overall interval:', result['overall']['ci'])
        print('hard delta:', result['tiers_descriptive']['hard']['delta'])
    else:
        print(result.get('registered_reading'))
else:
    print('comparison not available yet')


status: completed
registered outcome: material improvement
overall delta: 0.025477707006369428
overall interval: [0.012738853503184714, 0.03821656050955414]
hard delta: 0.045454545454545456


## 14. Close the units record

In [17]:
# record the balance after the session finishes
import json, os, time
cu_after = 553.49       # fill from the resources panel
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)


logged: {'session_id': '20260813_111301', 'ts_start': '2026-08-13 11:13', 'notebook': '031_colab_b7_n4_a2_record', 'colab_cu_before': 575.43, 'unit_rate': 6.77, 'ts_end': '2026-08-13 14:31', 'colab_cu_after': 553.49, 'colab_cu_used': 21.94, 'closed_on_resume': False}
